# Qualité des données d’accidents routiers

On exploite les fichiers BAAC 2023 et 2024 disponibles dans `dataset/`. L’objectif est de diagnostiquer les champs critiques, tracer les anomalies, contrôler le calcul et comparer le résultat au chiffre publié quand cette référence est fournie.

**Périmètre connu :** le projet ne contient pas le chiffre mensuel publié ni de manifeste de complétude des extractions. La décision de publication restera donc un NO-GO tant que ces éléments ne sont pas renseignés et rapprochés.


## 0. Chargement


In [9]:
from pathlib import Path
import pandas as pd

BASE = Path("dataset")
ANNEES = [2023, 2024]
FICHIERS = ["caract", "lieux", "vehicules", "usagers"]

def charger(annee, table):
    chemin = BASE / str(annee) / f"{table}-{annee}.csv"
    return pd.read_csv(chemin, sep=";", dtype="string", encoding="utf-8")

donnees = {
    (annee, table): charger(annee, table)
    for annee in ANNEES
    for table in FICHIERS
}

for (annee, table), df in donnees.items():
    print(f"{annee} / {table:10s} : {len(df):>7,} lignes, {df.shape[1]} colonnes")


2023 / caract     :  54,822 lignes, 15 colonnes
2023 / lieux      :  70,860 lignes, 18 colonnes
2023 / vehicules  :  93,585 lignes, 11 colonnes
2023 / usagers    : 125,789 lignes, 16 colonnes
2024 / caract     :  54,402 lignes, 15 colonnes
2024 / lieux      :  70,248 lignes, 18 colonnes
2024 / vehicules  :  92,678 lignes, 11 colonnes
2024 / usagers    : 125,187 lignes, 16 colonnes


## 1. Diagnostic

`caract` porte une ligne par accident ; `usagers` porte les personnes et leur gravité. Les codes sentinelles et les valeurs vides sont comptés comme non renseignés. Le diagnostic cible les clés, les composantes de date, la gravité, et profile aussi les autres tables pour repérer les défauts de couverture.


In [10]:
SENTINELLES = {"", "-1", "N/A", "NA", "null", "None"}
CHAMPS_CRITIQUES = {
    "caract": ["Num_Acc", "jour", "mois", "an"],
    "lieux": ["Num_Acc"],
    "vehicules": ["Num_Acc"],
    "usagers": ["Num_Acc", "grav"],
}

def profil(df):
    lignes = []
    for col in df.columns:
        valeurs = df[col].fillna("").str.strip()
        manquants = valeurs.isin(SENTINELLES)
        lignes.append({
            "colonne": col,
            "manquants_ou_codes": int(manquants.sum()),
            "part_%": round(100 * manquants.mean(), 2),
            "valeurs_distinctes": int(valeurs.nunique()),
        })
    return pd.DataFrame(lignes).sort_values(["part_%", "colonne"], ascending=[False, True])

for annee in ANNEES:
    print(f"\n--- {annee} : champs critiques ---")
    for table, champs in CHAMPS_CRITIQUES.items():
        df = donnees[(annee, table)]
        profil_cible = profil(df).set_index("colonne").reindex(champs).reset_index()
        profil_cible.insert(0, "table", table)
        profil_cible.insert(0, "annee", annee)
        display(profil_cible)
    print(f"--- {annee} : dix colonnes les plus incomplètes par table ---")
    for table in FICHIERS:
        print(table)
        display(profil(donnees[(annee, table)]).head(10))



--- 2023 : champs critiques ---


,annee,table,colonne,manquants_ou_codes,part_%,valeurs_distinctes
0,2023,caract,Num_Acc,0,0.0,54822
1,2023,caract,jour,0,0.0,31
2,2023,caract,mois,0,0.0,12
3,2023,caract,an,0,0.0,1


,annee,table,colonne,manquants_ou_codes,part_%,valeurs_distinctes
0,2023,lieux,Num_Acc,0,0.0,54822


,annee,table,colonne,manquants_ou_codes,part_%,valeurs_distinctes
0,2023,vehicules,Num_Acc,0,0.0,54822


,annee,table,colonne,manquants_ou_codes,part_%,valeurs_distinctes
0,2023,usagers,Num_Acc,0,0.00,54822
1,2023,usagers,grav,118,0.09,5


...

,colonne,manquants_ou_codes,part_%,valeurs_distinctes
12,lartpc,70215,99.95,16
4,v2,64332,91.58,28
13,larrout,48646,69.25,65
10,pr1,27432,39.05,1353
9,pr,27364,38.95,461
3,v1,16272,23.16,4
2,voie,13331,18.98,19738
5,circ,4354,6.20,5
6,nbv,4178,5.95,15
7,vosp,3832,5.45,5


vehicules


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
10,occutc,91729,98.98,24
9,motor,192,0.21,8
3,senc,68,0.07,5
7,choc,44,0.05,11
8,manv,27,0.03,28
5,obs,27,0.03,19
6,obsm,30,0.03,8
0,Num_Acc,0,0.00,54402
4,catv,1,0.00,32
1,id_vehicule,0,0.00,92678


usagers


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
15,etatp,114932,91.81,4
12,secu3,113133,90.37,11
13,locp,61771,49.34,11
14,actp,61726,49.31,13
11,secu2,53813,42.99,11
9,trajet,2626,2.10,8
8,an_nais,2579,2.06,106
7,sexe,2395,1.91,3
10,secu1,2103,1.68,11
0,Num_Acc,0,0.00,54402


In [11]:
for annee in ANNEES:
    car = donnees[(annee, "caract")].copy()
    car["Num_Acc"] = car["Num_Acc"].str.strip()
    accidents = set(car.loc[~car["Num_Acc"].isin(SENTINELLES), "Num_Acc"])
    cles_absentes = car["Num_Acc"].isna() | car["Num_Acc"].isin(SENTINELLES)
    print(f"\n{annee} :")
    print("Lignes caract :", len(car))
    print("Clés Num_Acc manquantes :", int(cles_absentes.sum()))
    print("Lignes en doublon Num_Acc :", int(car.loc[~cles_absentes, "Num_Acc"].duplicated(keep=False).sum()))
    for table in ["lieux", "vehicules", "usagers"]:
        df = donnees[(annee, table)].copy()
        df["Num_Acc"] = df["Num_Acc"].str.strip()
        cles = set(df.loc[~df["Num_Acc"].isin(SENTINELLES), "Num_Acc"])
        print(f"  {table:10s} | lignes={len(df):>7,} | clés sans caract={len(cles-accidents):>4} | accidents sans détail={len(accidents-cles):>4}")



2023 :
Lignes caract : 54822
Clés Num_Acc manquantes : 0
Lignes en doublon Num_Acc : 0
  lieux      | lignes= 70,860 | clés sans caract=   0 | accidents sans détail=   0
  vehicules  | lignes= 93,585 | clés sans caract=   0 | accidents sans détail=   0
  usagers    | lignes=125,789 | clés sans caract=   0 | accidents sans détail=   0

2024 :
Lignes caract : 54402
Clés Num_Acc manquantes : 0
Lignes en doublon Num_Acc : 0
  lieux      | lignes= 70,248 | clés sans caract=   0 | accidents sans détail=   0
  vehicules  | lignes= 92,678 | clés sans caract=   0 | accidents sans détail=   0
  usagers    | lignes=125,187 | clés sans caract=   0 | accidents sans détail=   0


La clé `Num_Acc`, la date et la gravité sont les champs les plus importants pour le calcul. Les autres tables servent surtout à décrire le contexte de l’accident.


## 2. Nettoyage et anomalies

Les espaces des clés sont normalisés dans les copies de travail. Les doublons d’accidents et dates invalides sont exclus du KPI. La gravité est traitée selon les codes BAAC : `2` tué, `3` hospitalisé, `4` blessé léger, `1` indemne. Pour l’indicateur « au moins un tué », on teste directement la présence du code `2` parmi les usagers ; on ne compare pas les codes comme s’ils formaient une échelle numérique.

Une gravité inconnue est tracée. L’accident reste calculable si au moins un usager a une gravité valide ; sinon, il est exclu et le motif est enregistré.


In [12]:
def preparer_annee(annee):
    car = donnees[(annee, "caract")].copy()
    usa = donnees[(annee, "usagers")].copy()
    anomalies = []

    car["Num_Acc"] = car["Num_Acc"].str.strip()
    usa["Num_Acc"] = usa["Num_Acc"].str.strip()
    car["_ligne_source"] = range(2, len(car) + 2)
    usa["_ligne_source"] = range(2, len(usa) + 2)

    doublon = car["Num_Acc"].duplicated(keep=False) | car["Num_Acc"].isin(SENTINELLES)
    for _, ligne in car.loc[doublon].iterrows():
        anomalies.append({"annee": annee, "regle": "clé accident absente ou dupliquée", "Num_Acc": ligne["Num_Acc"], "ligne_source": ligne["_ligne_source"], "action": "exclu du KPI", "statut": "à arbitrer"})

    dates = pd.to_datetime(
        car["an"].str.strip() + "-" + car["mois"].str.strip().str.zfill(2) + "-" + car["jour"].str.strip().str.zfill(2),
        errors="coerce",
    )
    dates_invalides = dates.isna() | (dates.dt.year != annee)
    for _, ligne in car.loc[dates_invalides].iterrows():
        anomalies.append({"annee": annee, "regle": "date invalide ou année incohérente", "Num_Acc": ligne["Num_Acc"], "ligne_source": ligne["_ligne_source"], "action": "exclu du mensuel", "statut": "à corriger à la source"})

    usa["grav"] = usa["grav"].str.strip()
    grav_valide = usa["grav"].isin(["1", "2", "3", "4"])
    for _, ligne in usa.loc[~grav_valide].iterrows():
        anomalies.append({"annee": annee, "regle": "gravité absente ou hors domaine", "Num_Acc": ligne["Num_Acc"], "ligne_source": ligne["_ligne_source"], "action": "usager ignoré; accident exclu si aucune gravité valide", "statut": "à vérifier"})

    usa_valide = usa.loc[grav_valide].copy()
    rang_severite = {"2": 1, "3": 2, "4": 3, "1": 4}
    usa_valide["rang_severite"] = usa_valide["grav"].map(rang_severite)
    grav_accident = usa_valide.groupby("Num_Acc")["rang_severite"].min()
    a_un_tue = usa_valide.assign(est_tue=usa_valide["grav"].eq("2")).groupby("Num_Acc")["est_tue"].any()

    incidents = pd.DataFrame({"Num_Acc": car["Num_Acc"], "date": dates})
    incidents["mois"] = incidents["date"].dt.to_period("M").astype("string")
    incidents["rang_severite"] = incidents["Num_Acc"].map(grav_accident)
    incidents["a_un_tue"] = incidents["Num_Acc"].map(a_un_tue).fillna(False).astype(bool)
    incidents["cle_unique"] = ~doublon
    incidents["date_valide"] = ~dates_invalides
    incidents["gravite_disponible"] = incidents["rang_severite"].notna()
    incidents["eligible"] = incidents["cle_unique"] & incidents["date_valide"] & incidents["gravite_disponible"]
    journal = pd.DataFrame(anomalies, columns=["annee", "regle", "Num_Acc", "ligne_source", "action", "statut"])
    return incidents, journal

incidents_par_an = {}
journal_par_an = {}
for annee in ANNEES:
    incidents_par_an[annee], journal_par_an[annee] = preparer_annee(annee)
    print(annee, "accidents chargés :", len(incidents_par_an[annee]), "| anomalies tracées :", len(journal_par_an[annee]))


2023 accidents chargés : 54822 | anomalies tracées : 118
2024 accidents chargés : 54402 | anomalies tracées : 0


Le journal conserve l’année, la règle enfreinte, la clé, la ligne du fichier source, l’action appliquée et le statut de traitement. Il trace les exclusions et les valeurs suspectes ; ce n’est pas un journal de correction à la source. Toute correction effective doit être confirmée puis documentée séparément avant le prochain calcul.


In [13]:
journal_corrections = pd.concat(journal_par_an.values(), ignore_index=True)
display(journal_corrections.head(20))
journal_corrections.to_csv("journal_corrections.csv", index=False, encoding="utf-8")
print("Impact des anomalies par année et règle :")
impact_anomalies = (
    journal_corrections.groupby(["annee", "regle", "action"], dropna=False)
    .agg(lignes_source=("ligne_source", "size"), accidents_concernes=("Num_Acc", "nunique"))
    .reset_index()
)
display(impact_anomalies)


,annee,regle,Num_Acc,ligne_source,action,statut
0,2023,gravité absente ou hors domaine,202300000421,951,usager ignoré; accident exclu si aucune gravit...,à vérifier
1,2023,gravité absente ou hors domaine,202300001254,2808,usager ignoré; accident exclu si aucune gravit...,à vérifier
2,2023,gravité absente ou hors domaine,202300001462,3284,usager ignoré; accident exclu si aucune gravit...,à vérifier
3,2023,gravité absente ou hors domaine,202300002015,4558,usager ignoré; accident exclu si aucune gravit...,à vérifier
4,2023,gravité absente ou hors domaine,202300002418,5446,usager ignoré; accident exclu si aucune gravit...,à vérifier
5,2023,gravité absente ou hors domaine,202300002966,6702,usager ignoré; accident exclu si aucune gravit...,à vérifier
6,2023,gravité absente ou hors domaine,202300003477,7843,usager ignoré; accident exclu si aucune gravit...,à vérifier
7,2023,gravité absente ou hors domaine,202300003552,8008,usager ignoré; accident exclu si aucune gravit...,à vérifier
8,2023,gravité absente ou hors domaine,202300004224,9596,usager ignoré; accident exclu si aucune gravit...,à vérifier
9,2023,gravité absente ou hors domaine,202300004694,10690,usager ignoré; accident exclu si aucune gravit...,à vérifier


Impact des anomalies par année et règle :


,annee,regle,action,lignes_source,accidents_concernes
0,2023,gravité absente ou hors domaine,usager ignoré; accident exclu si aucune gravit...,118,116


## 3. Contrôles avant publication

Les contrôles ci-dessous sont exécutés sur les données transformées : unicité des clés retenues, dates valides et cohérentes avec l’année, gravité exploitable pour chaque ligne éligible, et concordance entre l’indicateur « tué » et les codes usagers. Le taux d’exclusion est chiffré. La complétude temporelle d’une extraction et le rapprochement à une publication externe restent à confirmer avec leurs sources de référence.


## 4. Indicateur et décision

Le KPI principal est le nombre mensuel d’accidents corporels éligibles ; le complément compte les accidents ayant au moins un usager avec `grav == 2`. La comparaison 2023/2024 est faite sur les mêmes mois calendaires. Elle reste descriptive : la comparabilité des extractions doit être confirmée. Renseigner la référence publiée et confirmer la complétude des extractions pour obtenir un GO.


In [ ]:
toutes = pd.concat(incidents_par_an.values(), ignore_index=True)
eligibles = toutes.loc[toutes["eligible"]].copy()
eligibles["annee"] = eligibles["date"].dt.year
mensuel = (
    eligibles.groupby(["annee", "mois"], as_index=False)
    .agg(
        accidents_comptes=("Num_Acc", "nunique"),
        accidents_avec_tue=("a_un_tue", "sum"),
    )
)
mensuel["accidents_avec_tue"] = mensuel["accidents_avec_tue"].astype(int)

mensuel["mois_num"] = pd.PeriodIndex(mensuel["mois"], freq="M").month
comparaison = mensuel.pivot(index="mois_num", columns="annee", values="accidents_comptes")
comparaison.index.name = "mois_calendaire"
if all(a in comparaison.columns for a in [2023, 2024]):
    comparaison["ecart_2024_vs_2023"] = comparaison[2024] - comparaison[2023]
    comparaison["ecart_%"] = (100 * comparaison["ecart_2024_vs_2023"] / comparaison[2023]).round(1)

display(mensuel.sort_values(["annee", "mois"]).drop(columns="mois_num"))
display(comparaison)


,annee,mois,accidents_comptes,accidents_avec_tue
0,2023,2023-01,4053,232
1,2023,2023-02,3682,219
2,2023,2023-03,3998,212
3,2023,2023-04,4162,224
4,2023,2023-05,4767,250
5,2023,2023-06,5452,287
6,2023,2023-07,4754,304
7,2023,2023-08,4121,287
8,2023,2023-09,5161,299
9,2023,2023-10,5389,311


annee,2023,2024,ecart_2024_vs_2023,ecart_%
mois_calendaire,,,,
1,4053,4069,16,0.4
2,3682,3792,110,3.0
3,3998,4370,372,9.3
4,4162,4586,424,10.2
5,4767,4517,-250,-5.2
6,5452,5135,-317,-5.8
7,4754,5209,455,9.6
8,4121,4450,329,8.0
9,5161,4620,-541,-10.5


In [15]:
resume = []
for annee, df in incidents_par_an.items():
    source = donnees[(annee, "caract")]
    anomalies_gravite = journal_par_an[annee].query("regle == 'gravité absente ou hors domaine'")
    resume.append({
        "annee": annee,
        "lignes_caract": len(source),
        "cles_uniques_non_vides": source["Num_Acc"].nunique(),
        "dates_valides": int(df["date_valide"].sum()),
        "accidents_avec_gravite": int(df["gravite_disponible"].sum()),
        "accidents_eligibles": int(df["eligible"].sum()),
        "exclus_du_kpi": int((~df["eligible"]).sum()),
        "usagers_gravite_invalide": len(anomalies_gravite),
        "accidents_touches_gravite_invalide": anomalies_gravite["Num_Acc"].nunique(),
        "anomalies_tracees": len(journal_par_an[annee]),
        "premier_mois_present": df["mois"].dropna().min(),
        "dernier_mois_present": df["mois"].dropna().max(),
    })
resume = pd.DataFrame(resume)
display(resume)

for annee, df in incidents_par_an.items():
    elig = df.loc[df["eligible"]]
    assert elig["Num_Acc"].notna().all(), f"{annee}: clé manquante parmi les éligibles"
    assert elig["Num_Acc"].is_unique, f"{annee}: clé dupliquée parmi les éligibles"
    assert elig["date"].notna().all() and (elig["date"].dt.year == annee).all(), f"{annee}: date invalide"
    assert elig["rang_severite"].notna().all(), f"{annee}: gravité absente parmi les éligibles"
    usa = donnees[(annee, "usagers")].copy()
    usa["Num_Acc"] = usa["Num_Acc"].str.strip()
    tue_attendu = usa.loc[usa["grav"].str.strip().eq("2")].groupby("Num_Acc").size().gt(0)
    indicateur_attendu = df["Num_Acc"].isin(tue_attendu[tue_attendu].index)
    assert df["a_un_tue"].equals(indicateur_attendu), f"{annee}: indicateur tué différent des lignes usagers"
print("Contrôles automatiques : OK")


,annee,lignes_caract,cles_uniques_non_vides,dates_valides,accidents_avec_gravite,accidents_eligibles,exclus_du_kpi,usagers_gravite_invalide,accidents_touches_gravite_invalide,anomalies_tracees,premier_mois_present,dernier_mois_present
0,2023,54822,54822,54822,54822,54822,0,118,116,118,2023-01,2023-12
1,2024,54402,54402,54402,54402,54402,0,0,0,0,2024-01,2024-12


Contrôles automatiques : OK


### Décision

Le chiffre publié doit être renseigné par `année-mois`, et la complétude des extractions doit être confirmée à partir du manifeste ou du producteur des données. Sans ces deux éléments, la décision reste **NO-GO**. Les dates minimale et maximale ne prouvent pas à elles seules que les fichiers sont complets.

Dans la cellule suivante, renseigner la référence officielle et passer `EXTRACTIONS_CONFIRMEES_COMPLETES` à `True` uniquement après vérification documentée.


In [ ]:
REFERENCE_PUBLIEE = {} 
EXTRACTIONS_CONFIRMEES_COMPLETES = False 

if REFERENCE_PUBLIEE:
    ref = pd.Series(REFERENCE_PUBLIEE, name="accidents_publies").rename_axis("mois").reset_index()
    mensuel_ref = mensuel[["mois", "accidents_comptes"]].copy()
    rapprochement = ref.merge(mensuel_ref, on="mois", how="outer", indicator=True)
    rapprochement["ecart"] = rapprochement["accidents_comptes"] - rapprochement["accidents_publies"]
    display(rapprochement.sort_values("mois"))
    rapprochement_ok = rapprochement["_merge"].eq("both").all() and rapprochement["ecart"].eq(0).all()
else:
    rapprochement_ok = False
    print("Rapprochement impossible : renseigner la référence officielle dans REFERENCE_PUBLIEE.")

go = rapprochement_ok and EXTRACTIONS_CONFIRMEES_COMPLETES
print("Décision :", "GO" if go else "NO-GO — référence, complétude ou concordance à confirmer")


Rapprochement impossible : renseigner la référence officielle dans REFERENCE_PUBLIEE.
Décision : NO-GO — référence, complétude ou concordance à confirmer
